# Preprocess

In [ ]:
import pandas as pd

data = pd.read_csv("twitter_data.csv", header=None,
                   encoding='latin-1' # for tiktok data
                   )

data

FileNotFoundError: [Errno 2] No such file or directory: 'twitter_data.csv'

In [ ]:
data.columns = ['text']

data

In [ ]:
import string
import re
import nltk
# nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

data['text'] = data['text'].str.lower() # lowercase

for index, sentence in enumerate(data['text']):
    sentence = re.sub(r'\n', ' ', sentence) # remove newline
    sentence = re.sub(r'https\S+', '', sentence) # remove URL
    sentence = re.sub(r'http\S+', '', sentence) # remove URL
    sentence = re.sub(r'@\S+', '', sentence) # remove mentions
    sentence = re.sub(r"\d+", "", sentence) # remove number
    sentence = sentence.translate(str.maketrans("","",string.punctuation)) # remove punctuations
    sentence = sentence.strip() # remove whitespaces
    sentence = re.compile('[^a-zA-Z ]').sub('', sentence) # remove emojis
    # tokens = nltk.tokenize.word_tokenize(sentence) # tokenize
    # data.loc[index, 'text'] = tokens
    data.loc[index, 'text'] = sentence

i = 0

for index, s in enumerate(data.text):
  if s == "":
    data.drop(index, inplace=True)
    i += 1

print(i)
print(data)

In [ ]:
data.to_csv('preprocessed_twitter_data.csv', index=False)

# Transfer Learning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("ridho2401/sa-tapera")
model = AutoModelForSequenceClassification.from_pretrained("ridho2401/sa-tapera")

In [ ]:
import pandas as pd

df = pd.read_csv("manually_labeled_tiktok_data.csv", header=None)
df.columns = ['text', 'label']

df2 = pd.read_csv("manually_labeled_instagram_data.csv", header=None)
df2.columns = ['text', 'label']

df3 = pd.read_csv("manually_labeled_twitter_data.csv", header=None)
df3.columns = ['text', 'label']

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['label'].map(label2id)
df2['label'] = df2['label'].map(label2id)
df3['label'] = df3['label'].map(label2id)

df = pd.concat([df, df2, df3], ignore_index=True)

In [ ]:
from sklearn.model_selection import train_test_split

# 60% train, 40% val + test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.4, random_state=42, stratify=df['label'] # balancing
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

In [ ]:
# tokenizing

train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True)
val_encodings = tokenizer(val_texts.tolist(), truncation=True, padding=True)
test_encodings = tokenizer(test_texts.tolist(), truncation=True, padding=True)

In [ ]:
import torch

class dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    # to stop looping
    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = dataset(train_encodings, train_labels)
val_dataset = dataset(val_encodings, val_labels)
test_dataset = dataset(test_encodings, test_labels)

In [ ]:
import wandb
wandb.init(project="huggingface", name="transfer_learning")

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

# metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

# training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="transfer_learning",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3.5e-5,
    weight_decay=0.025,
    seed=42
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
def model_init():
    return model  # reuse the same model or define a fresh copy

def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [4, 8])
    }

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
df['label'].value_counts()

,count
label,
0,943
1,488
2,225


In [ ]:
best_run = trainer.hyperparameter_search(
    direction="maximize",  # optimize for best accuracy or f1
    n_trials=10,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_f1"]
)

print("Best hyperparameters found:")
print(best_run)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
[I 2025-05-10 04:27:14,787] A new study created in memory with name: no-name-270a8137-2f50-49bf-8ba0-a67095b06889


Step,Training Loss
500,0.697300


[I 2025-05-10 04:29:05,140] Trial 0 finished with value: 0.7390096783978634 and parameters: {'learning_rate': 2.0746272294462437e-05, 'num_train_epochs': 3, 'weight_decay': 0.2133202118819601, 'lr_scheduler_type': 'linear', 'per_device_train_batch_size': 4}. Best is trial 0 with value: 0.7390096783978634.


Step,Training Loss
500,0.195800
1000,0.067800


[I 2025-05-10 04:32:32,820] Trial 1 finished with value: 0.7403176982324036 and parameters: {'learning_rate': 3.368000503143269e-05, 'num_train_epochs': 6, 'weight_decay': 0.025239915519423218, 'lr_scheduler_type': 'linear', 'per_device_train_batch_size': 4}. Best is trial 1 with value: 0.7403176982324036.


...

Step,Training Loss
500,0.000000


[I 2025-05-10 04:51:37,276] Trial 8 finished with value: 0.7200952393339325 and parameters: {'learning_rate': 2.495972455818338e-05, 'num_train_epochs': 4, 'weight_decay': 0.27362883365603174, 'lr_scheduler_type': 'linear', 'per_device_train_batch_size': 4}. Best is trial 1 with value: 0.7403176982324036.


Step,Training Loss
500,0.004300


[I 2025-05-10 04:53:23,743] Trial 9 finished with value: 0.7113379215357052 and parameters: {'learning_rate': 1.6095455119654953e-05, 'num_train_epochs': 4, 'weight_decay': 0.02346120605555154, 'lr_scheduler_type': 'cosine', 'per_device_train_batch_size': 8}. Best is trial 1 with value: 0.7403176982324036.


Best hyperparameters found:
BestRun(run_id='1', objective=0.7403176982324036, hyperparameters={'learning_rate': 3.368000503143269e-05, 'num_train_epochs': 6, 'weight_decay': 0.025239915519423218, 'lr_scheduler_type': 'linear', 'per_device_train_batch_size': 4}, run_summary=None)


In [ ]:
trainer.train()

Step,Training Loss
500,0.731200
1000,0.257300


TrainOutput(global_step=1494, training_loss=0.3597958474114557, metrics={'train_runtime': 214.8677, 'train_samples_per_second': 27.729, 'train_steps_per_second': 6.953, 'total_flos': 391907435705856.0, 'train_loss': 0.3597958474114557, 'epoch': 6.0})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
text = "makasih"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=1).item()
print(predicted_class)

1


In [ ]:
text = "amminn"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=1).item()
print(predicted_class)

1


In [ ]:
text = "payah"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=1).item()
print(predicted_class)

1


In [ ]:
text = "keren"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=1).item()
print(predicted_class)

2


In [ ]:
text = "indonesia keren"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=1).item()
print(predicted_class)

2


In [ ]:
trainer.evaluate(test_dataset)

{'eval_loss': 1.4606012105941772,
 'eval_accuracy': 0.75,
 'eval_f1': 0.7454345842901633,
 'eval_runtime': 2.1784,
 'eval_samples_per_second': 152.405,
 'eval_steps_per_second': 38.101,
 'epoch': 6.0}

In [ ]:
trainer.save_model("./transfer_learning_result")
tokenizer.save_pretrained("./transfer_learning_result")

('./transfer_learning_result/tokenizer_config.json',
 './transfer_learning_result/special_tokens_map.json',
 './transfer_learning_result/vocab.txt',
 './transfer_learning_result/added_tokens.json',
 './transfer_learning_result/tokenizer.json')

In [ ]:
!zip -r /content/transfer_learning_result.zip /content/transfer_learning_result

  adding: content/transfer_learning_result/ (stored 0%)
  adding: content/transfer_learning_result/config.json (deflated 51%)
  adding: content/transfer_learning_result/model.safetensors (deflated 7%)
  adding: content/transfer_learning_result/training_args.bin (deflated 51%)
  adding: content/transfer_learning_result/vocab.txt (deflated 52%)
  adding: content/transfer_learning_result/special_tokens_map.json (deflated 80%)
  adding: content/transfer_learning_result/tokenizer.json (deflated 71%)
  adding: content/transfer_learning_result/tokenizer_config.json (deflated 74%)


In [ ]:
from google.colab import files
files.download("/content/transfer_learning_result.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Sentiment Auto Label

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("./transfer_learning_result")
model = AutoModelForSequenceClassification.from_pretrained("./transfer_learning_result")

In [ ]:
print(model.config)

BertConfig {
  "_attn_implementation_autoset": true,
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_ids": 0,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "NEGATIVE",
    "1": "NEUTRAL",
    "2": "POSITIVE"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "NEGATIVE": 0,
    "NEUTRAL": 1,
    "POSITIVE": 2
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "torch_dtype": "float32",
  "transformers_version": "4.51.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 31923
}



In [ ]:
df = pd.read_csv("manually_labeled_tiktok_data.csv", header=None)
df.columns = ['text', 'label']

df2 = pd.read_csv("manually_labeled_instagram_data.csv", header=None)
df2.columns = ['text', 'label']

df3 = pd.read_csv("manually_labeled_twitter_data.csv", header=None)
df3.columns = ['text', 'label']

In [ ]:
# Function to predict sentiment
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    label = torch.argmax(probs, dim=1).item()

    label_map = {2: "positive", 1: "neutral", 0: "negative"}
    return label_map[label]

df['sentiment'] = df['text'].apply(predict_sentiment)
df2['sentiment'] = df2['text'].apply(predict_sentiment)
df3['sentiment'] = df3['text'].apply(predict_sentiment)

In [ ]:
df.to_csv("labeled_tflearning_tiktok_data.csv", index=False, header=False)
df2.to_csv("labeled_tflearning_instagram_data.csv", index=False, header=False)
df3.to_csv("labeled_tflearning_twitter_data.csv", index=False, header=False)